In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from gwosc.locate import get_run_urls
from gwpy.time import from_gps, to_gps

sys.path.append("../src")
from busqueda import dentro, duracion, interseccion, segmentos_de_analisis
from datos import eventos_catalogo, segmentos

In [ ]:
DATOS = Path("../datos")
MES = ("2019-05-01", "2019-06-01")
CONJUNTO = "O3a"
CATALOGO = "GWTC-2.1-confident"
OTROS_CATALOGOS = ["GWTC-2.1-marginal", "IAS-O3a", "O3_IMBH_marginal"]
FLAGS = ["DATA", "CBC_CAT1", "NO_CBC_HW_INJ"]
FLAG_VETO = "CBC_CAT2"
RECORTE_BLOQUE = 8
DURACION_SEGMENTO = 256
BORDE_INICIAL = 16
BORDE_FINAL = 4
DURACION_ARCHIVO = 4096
MB_POR_ARCHIVO = 130
MISMO_EVENTO = 1
DIA = 86400

In [ ]:
inicio, fin = int(to_gps(MES[0])), int(to_gps(MES[1]))
buenos = [(inicio, fin)]
for ifo in ["H1", "L1"]:
    for flag in FLAGS:
        buenos = interseccion(buenos, segmentos(f"{ifo}_{flag}", inicio, fin, DATOS))
bloques = [(a + RECORTE_BLOQUE, b - RECORTE_BLOQUE) for a, b in buenos if b - a >= DURACION_SEGMENTO + 2 * RECORTE_BLOQUE]
analisis = [(a + BORDE_INICIAL, b - BORDE_FINAL) for a, b in bloques]
for ifo in ["H1", "L1"]:
    analisis = interseccion(analisis, segmentos(f"{ifo}_{FLAG_VETO}", inicio, fin, DATOS))
tramos = [t for a, b in bloques for t in segmentos_de_analisis(a, b, DURACION_SEGMENTO, BORDE_INICIAL, BORDE_FINAL)]
largos = np.array([b - a for a, b in bloques])
print(f"Mayo de 2019: GPS {inicio} a {fin} ({(fin - inicio) / DIA:.0f} días)")
print(f"H1 y L1 con {', '.join(FLAGS)}: {duracion(buenos) / DIA:.2f} días en {len(buenos)} tramos")
print(f"Bloques de al menos {DURACION_SEGMENTO + 2 * RECORTE_BLOQUE} s: {len(bloques)}, {duracion(bloques) / DIA:.2f} días (se descartan {len(buenos) - len(bloques)} tramos más cortos); mediana {np.median(largos) / 3600:.1f} h, el más largo {largos.max() / 3600:.1f} h")
print(f"Tiempo analizado (sin bordes ni vetos {FLAG_VETO}): {duracion(analisis) / DIA:.2f} días en {len(analisis)} tramos")
print(f"Segmentos de {DURACION_SEGMENTO} s por detector: {len(tramos)}")

In [ ]:
eventos = eventos_catalogo(CATALOGO, inicio, fin, DATOS)
gps = np.array([e["GPS"] for e in eventos])
con_datos = dentro(gps, buenos)
analizados = dentro(gps, analisis)
masas_detector = np.array([e["chirp_mass_source"] * (1 + e["redshift"]) for e in eventos])
for e, masa, d, a in zip(eventos, masas_detector, con_datos, analizados):
    print(f"{e['commonName']:16} {from_gps(e['GPS']):%d/%m %H:%M} UTC, GPS {e['GPS']}: Mc en el detector ≈{masa:5.1f}, SNR de red {e['network_matched_filter_snr']:4.1f}, FAR {e['far']:.1e} por año, p_astro {e['p_astro']:.2f}; H1 y L1 {'sí' if d else 'no'}, analizado {'sí' if a else 'no'}")
print(f"{CATALOGO}: {len(eventos)} eventos, {con_datos.sum()} con H1 y L1 buenos, {analizados.sum()} en el tiempo analizado")
print(f"Masa chirp en el detector de los analizados: {masas_detector[analizados].min():.1f} a {masas_detector[analizados].max():.1f}")

In [ ]:
for nombre in OTROS_CATALOGOS:
    otros = eventos_catalogo(nombre, inicio, fin, DATOS)
    nuevos = [e for e in otros if np.abs(gps - e["GPS"]).min() > MISMO_EVENTO]
    print(f"{nombre}: eventos en el mes {len(otros)}, fuera de {CATALOGO} {len(nuevos)}")
    for e in nuevos:
        print(f"    {e['commonName']}, GPS {e['GPS']}, SNR de red {e['network_matched_filter_snr']}, analizado {'sí' if dentro(np.array([e['GPS']]), analisis)[0] else 'no'}")

In [ ]:
archivos = sorted({s for a, b in bloques for s in range(a // DURACION_ARCHIVO * DURACION_ARCHIVO, b, DURACION_ARCHIVO)})
cruzan = sum(a // DURACION_ARCHIVO != (b - 1) // DURACION_ARCHIVO for a, b in bloques)
urls = {ifo: get_run_urls(CONJUNTO, ifo, inicio, fin) for ifo in ["H1", "L1"]}
for ifo, lista in urls.items():
    disponibles = {int(u.split("-")[-2]) for u in lista}
    conjuntos = sorted({u.split("/")[-3] for u in lista})
    print(f"{ifo}: {len(lista)} archivos en GWOSC ({', '.join(conjuntos)}); de los {len(archivos)} que se necesitan faltan {len(set(archivos) - disponibles)}")
print(f"Se necesitan {len(archivos)} archivos de {DURACION_ARCHIVO} s por detector: ~{2 * len(archivos) * MB_POR_ARCHIVO / 1000:.0f} GB entre H1 y L1")
print(f"Bloques que cruzan el límite entre archivos: {cruzan} de {len(bloques)}")
print("Ejemplo:", urls["H1"][0])

In [ ]:
fig, eje = plt.subplots(figsize=(10, 2.5))
eje.broken_barh([((a - inicio) / DIA, (b - a) / DIA) for a, b in analisis], (0, 1), color="tab:blue")
for t, analizado in zip(gps, analizados):
    eje.axvline((t - inicio) / DIA, color="tab:red" if analizado else "gray", linewidth=1)
eje.set_xlim(0, (fin - inicio) / DIA)
eje.set_yticks([])
eje.set_xlabel("Días desde el 1 de mayo de 2019");